# Basketball Play Lab: free-first cloud training
Human labels stay authoritative. This notebook trains ball/rim detection and a small shot-result classifier; foul/turnover remain manual. No accuracy claim follows merely from a successful run.

**Before starting locally:** export a reviewed dataset; zip its entire version folder. Make a source zip containing only `playlab/`, `pyproject.toml`, and `uv.lock`. Exclude `.venv`, `.git`, videos, private keys and tokens. Compute each zip's SHA-256 (`Get-FileHash PATH -Algorithm SHA256` on Windows). Put both zips in Google Drive. A source zip hash pins the exact code even before a repository exists.

**Runtime:** choose a GPU runtime with Python 3.12. Free GPU access is not guaranteed. Drive keeps checkpoints after runtime expiry; a free session can stop while your laptop sleeps. There are no keep-alive tricks, API secrets, or paid GPU fallback in this notebook.

**Order:** configure → verify/extract → install → GPU check → checkpoint smoke → resume → train classifier. Stop before long training if any gate fails. Keep held-out test results unopened until the Day 3 freeze.

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount('/content/drive')
SOURCE_ZIP = Path('/content/drive/MyDrive/PlayLab/source.zip')
DATASET_ZIP = Path('/content/drive/MyDrive/PlayLab/dataset.zip')
SOURCE_SHA256 = 'PASTE_LOCAL_SOURCE_ZIP_SHA256'
DATASET_SHA256 = 'PASTE_LOCAL_DATASET_ZIP_SHA256'
RUN = Path('/content/drive/MyDrive/PlayLab/runs/pilot-01')
MAX_HOURS = 1.0  # Each run is capped at <=2 hours; stop sooner when needed.
EPOCHS = 40
assert 0 < MAX_HOURS <= 2


In [ ]:
import hashlib
import json
import sys
import zipfile

assert sys.version_info[:2] == (3, 12), 'Use a Python 3.12 runtime for this locked project.'
def extract_verified(archive, expected, target):
    assert len(expected) == 64, 'Paste the SHA-256 computed locally first.'
    digest = hashlib.file_digest(archive.open('rb'), 'sha256').hexdigest()
    assert digest.lower() == expected.lower(), 'Archive changed; do not continue.'
    target.mkdir(parents=True, exist_ok=True)
    assert not any(target.iterdir()), 'Use a new empty extraction folder.'
    with zipfile.ZipFile(archive) as bundle:
        for member in bundle.infolist():
            assert (target / member.filename).resolve().is_relative_to(target.resolve()), 'Unsafe zip path.'
            assert not ((member.external_attr >> 16) & 0o170000 == 0o120000), 'Symlinks are not accepted.'
        bundle.extractall(target)
    return target
code_root = extract_verified(SOURCE_ZIP, SOURCE_SHA256, Path('/content/playlab-source'))
data_root = extract_verified(DATASET_ZIP, DATASET_SHA256, Path('/content/playlab-dataset'))
projects = list(code_root.rglob('pyproject.toml'))
manifests = list(data_root.rglob('manifest.json'))
assert len(projects) == len(manifests) == 1, 'Each zip must contain exactly one project/dataset.'
PROJECT, DATASET = projects[0].parent, manifests[0].parent
RUN.mkdir(parents=True, exist_ok=True)
(RUN / 'archive-provenance.json').write_text(json.dumps({'source_sha256': SOURCE_SHA256, 'dataset_zip_sha256': DATASET_SHA256}, indent=2))
print('Archive hashes verified. Source videos are not required for training.')


In [ ]:
import subprocess

subprocess.run([sys.executable, '-m', 'pip', 'install', 'uv'], check=True)
subprocess.run(['uv', 'sync', '--locked', '--extra', 'vision', '--no-dev'], cwd=PROJECT, check=True)
PYTHON = PROJECT / '.venv/bin/python'
def run_code(source):
    subprocess.run([str(PYTHON), '-c', source], cwd=PROJECT, check=True)
run_code('import torch; print(torch.__version__); assert torch.cuda.is_available(), "No free GPU: stop and use CPU review/baseline locally."; print(torch.cuda.get_device_name(0))')
run_code(f'from playlab.dataset import verify_dataset; print(verify_dataset({str(DATASET)!r})["dataset_hash"])')
print('GPU and dataset checks passed; no training has started.')


## First checkpoint smoke
This intentionally interrupts a real detector run after one saved epoch. Inspect the persisted `last.pt` and RNG sidecar. The next cell resumes this same candidate rather than starting a new model. This verifies the resume path; it does not establish mathematical equivalence or basketball accuracy.
If a runtime later expires, remount Drive, verify/extract the same archives in fresh runtime folders, and rerun installation/GPU checks. Skip the smoke cell and run the resume cell against the existing run. Dataset and source hashes must match. Do not replace dataset labels inside an existing run.

In [ ]:
DETECTOR_RUN = RUN / 'ball-rim'
run_code(f'from playlab.training import train_detector; print(train_detector({str(DATASET)!r}, {str(DETECTOR_RUN)!r}, device="0", max_hours={MAX_HOURS}, epochs={EPOCHS}, stop_after_epochs=1))')
CHECKPOINT = DETECTOR_RUN / 'detector/weights/last.pt'
assert CHECKPOINT.is_file() and Path(str(CHECKPOINT) + '.rng.pt').is_file()
print('Checkpoint smoke saved to Drive. Ready for bounded resume.')


In [ ]:
DETECTOR_RUN = RUN / 'ball-rim'
CHECKPOINT = DETECTOR_RUN / 'detector/weights/last.pt'
run_code(f'from playlab.training import train_detector; print(train_detector({str(DATASET)!r}, {str(DETECTOR_RUN)!r}, device="0", resume={str(CHECKPOINT)!r}, max_hours={MAX_HOURS}, epochs={EPOCHS}))')
print('Read training.json and detector/results.csv in Drive. These are training/validation results only.')


In [ ]:
CLASSIFIER_RUN = RUN / 'shot-result'
run_code(f'from playlab.training import train_classifier; print(train_classifier({str(DATASET)!r}, {str(CLASSIFIER_RUN)!r}))')
print('Inspect training.json: selection uses validation only and retains the geometry baseline when it wins/ties.')
print('Download best.pt, classifier.joblib, all training/provenance JSON and validation reports for local review.')


## Freeze before the locked test
By the end of Day 2, freeze the code, dataset, detector weights, classifier bundle and analysis configuration. Run the locally documented `freeze` / `evaluate` commands once on Day 3. Include every eligible held-out shot, unknown result and missed detection in the appropriate denominator. Keep automatic and human-release-anchored analyses distinguishable.

Do not tune thresholds after opening test results. If labels/source/GPU access fails, deliver the reviewed possession editor and CPU baseline with explicit missing-training limitations. Free cloud runtime availability and phone notification delivery are separate capabilities; this notebook neither guarantees nor configures them.